# 10 · Silver · Unificação dos canais
Junta os canais numa estrutura única (dimensões + fato), base de todas as tabelas gold.
Canais incluídos agora: **loja física (ERP)** e **marketplace (Olist)**. O site entra quando tiver silver (notebooks 08 e 09).

- **Unificação por categoria e grupo, sem SKU único (D12):** `produto_id` guarda o código de origem de cada canal.
- **Nada é removido (D07):** a fato carrega a coluna `valido`, que resume as regras de cada canal. O filtro acontece só na gold.

| Canal | Regra de `valido` |
|---|---|
| `loja_fisica` | `valido` da `silver.erp_vendas` (D18) |
| `marketplace` | `valido AND incluir AND periodo_completo` da `silver.marketplace_itens` (D07–D09) |

**Equivalente no BigQuery:** mesma estrutura. Diferenças de sintaxe indicadas em cada célula.

### 1. `dim_categoria`
As 9 categorias do catálogo, com grupo e **margem**. A margem é calculada do próprio catálogo (`1 − custo ÷ preço`), que foi gerado com as margens do `parametros.yaml`: nenhum número novo no código, e a fonte continua sendo uma só.
O marketplace usa essa margem para **estimar** o custo, porque o Olist não tem custo.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.dim_categoria AS
SELECT categoria,
       grupo,
       CAST(ROUND(1 - SUM(custo) / SUM(preco_tabela), 2) AS DECIMAL(4,2)) AS margem
FROM horizonte.silver.erp_produtos
GROUP BY categoria, grupo;

### 2. `dim_canal` e `dim_loja`
O canal `site` já entra na dimensão, mesmo sem dados ainda.

**BigQuery:** `FROM VALUES (...) AS t(...)` vira `FROM UNNEST([STRUCT('loja_fisica' AS canal, 'Loja física' AS nome, FALSE AS online), ...])`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.dim_canal AS
SELECT *
FROM VALUES ('loja_fisica', 'Loja física', FALSE),
            ('site',        'Site',        TRUE),
            ('marketplace', 'Marketplace', TRUE)
     AS t(canal, nome, online);

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.dim_loja AS
SELECT loja_id, nome, cidade, uf, porte, 'loja_fisica' AS canal
FROM horizonte.silver.erp_lojas;

### 3. `fato_vendas`
Uma linha por item de venda (linha de cupom no ERP, unidade no Olist).
- **Receita do marketplace sem frete:** frete não é venda de produto e inflaria receita e margem.
- **Custo:** real no ERP (`custo_total`); estimado no marketplace (`receita × (1 − margem)`), sinalizado por `custo_estimado`.
- **Cliente (`cliente_id`):** nulo no ERP (cupom sem identificação); `customer_unique_id` no marketplace (D11). Base da recomendação (D19), que vai usar o cliente do site.
- **Categoria:** no marketplace vem do de-para (D04); `Não classificado` fica sem grupo, e já tem `incluir = false`.

**BigQuery:** `CAST(... AS STRING)` e `DECIMAL` → `NUMERIC`; `data_hora` como `DATETIME`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.fato_vendas AS
SELECT 'loja_fisica'           AS canal,
       v.data,
       v.data_hora,
       v.cupom_id              AS pedido_id,
       v.loja_id,
       CAST(NULL AS STRING)    AS cliente_id,   -- cupom fiscal sem identificação do cliente
       v.sku                   AS produto_id,
       v.categoria,
       v.grupo,
       v.quantidade,
       v.valor_total           AS receita,
       v.custo_total           AS custo,
       FALSE                   AS custo_estimado,
       v.valido
FROM horizonte.silver.erp_vendas v

UNION ALL

SELECT 'marketplace',
       CAST(i.data_compra AS DATE),
       i.data_compra,
       i.pedido_id,
       CAST(NULL AS STRING),
       p.cliente_id,                            -- customer_unique_id (D11)
       i.produto_id,
       i.categoria,
       c.grupo,
       i.quantidade,
       i.preco,
       CAST(i.preco * (1 - c.margem) AS DECIMAL(12,2)),
       TRUE,
       i.valido AND i.incluir AND i.periodo_completo
FROM horizonte.silver.marketplace_itens i
JOIN horizonte.silver.marketplace_pedidos p ON p.pedido_id = i.pedido_id
LEFT JOIN horizonte.silver.dim_categoria c ON c.categoria = i.categoria;

### 4. `dim_data`
Um dia por linha, do primeiro dia com venda válida até o fim do ano da última venda. Assim o calendário cobre a **Black Friday de 2026**, que é a data a prever.
- **Semana começa na segunda.**
- **Black Friday por regra, sem data fixa no código:** sexta-feira de novembro entre os dias 23 e 29 (o dia seguinte ao Dia de Ação de Graças americano).

**BigQuery:** `explode(sequence(...))` → `UNNEST(GENERATE_DATE_ARRAY(...))`; `date_trunc('WEEK', data)` → `DATE_TRUNC(data, ISOWEEK)`; `weekday(data)` (0 = segunda) → `MOD(EXTRACT(DAYOFWEEK FROM data) + 5, 7)`; `bool_or` → `LOGICAL_OR`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.dim_data AS
WITH limites AS (
  SELECT MIN(data) AS inicio,
         make_date(year(MAX(data)), 12, 31) AS fim
  FROM horizonte.silver.fato_vendas
  WHERE valido
),
dias AS (
  SELECT explode(sequence(inicio, fim, INTERVAL 1 DAY)) AS data
  FROM limites
),
base AS (
  SELECT data,
         year(data)                              AS ano,
         month(data)                             AS mes,
         weekday(data) + 1                       AS dia_semana,   -- 1 = segunda ... 7 = domingo
         CAST(date_trunc('WEEK', data) AS DATE)  AS semana,
         month(data) = 11 AND weekday(data) = 4
           AND day(data) BETWEEN 23 AND 29       AS black_friday
  FROM dias
)
SELECT *,
       bool_or(black_friday) OVER (PARTITION BY semana) AS semana_black_friday
FROM base;

### 5. Conferência
Os totais da fato precisam bater com a silver de origem: o ERP com a D18 e o marketplace com os 42.758 itens da silver do Olist.

In [0]:
SELECT canal,
       COUNT(*)                                     AS linhas,    -- loja_fisica 401.335 · marketplace 112.650
       COUNT_IF(valido)                             AS validas,   -- loja_fisica 399.736 · marketplace 42.758
       SUM(CASE WHEN valido THEN quantidade END)    AS unidades,
       SUM(CASE WHEN valido THEN receita END)       AS receita,   -- loja_fisica 30.795.090,78
       COUNT_IF(valido AND grupo IS NULL)           AS validas_sem_grupo  -- 0 nos dois
FROM horizonte.silver.fato_vendas
GROUP BY canal
ORDER BY canal;

In [0]:
SELECT 'dim_categoria' AS tabela, COUNT(*) AS linhas FROM horizonte.silver.dim_categoria  -- 9
UNION ALL SELECT 'dim_canal', COUNT(*) FROM horizonte.silver.dim_canal                   -- 3
UNION ALL SELECT 'dim_loja', COUNT(*) FROM horizonte.silver.dim_loja                     -- 12
UNION ALL SELECT 'dim_data', COUNT(*) FROM horizonte.silver.dim_data                     -- 726 se o 1º dia for 05/01/2025
UNION ALL SELECT 'dias de Black Friday', COUNT_IF(black_friday) FROM horizonte.silver.dim_data;  -- 2 (28/11/2025 e 27/11/2026)

In [0]:
SELECT MIN(data) AS primeiro_dia,   -- 2025-01-05
       MAX(data) AS ultimo_dia,     -- 2026-12-31
       collect_list(data) FILTER (WHERE black_friday) AS black_fridays
FROM horizonte.silver.dim_data;